# ATHLLM — Kaggle 4B Distillation & Validation

This notebook is the low-cost experimental entry point for ATHLLM. It is designed for a constrained Kaggle GPU: validate the Spark-X2.5-inspired architecture, load an authorized teacher checkpoint locally, build/inspect a verified distillation manifest, and run student experiments.

**Do not put proprietary weights or the 5T-token corpus in the notebook.** Download authorized checkpoints/datasets at runtime and keep secrets in Kaggle Secrets.


## Experiment ladder
1. Architecture smoke test
2. Teacher-data inspection
3. Distillation/SFT
4. Verifiable reasoning + coding RL
5. Int4 quantization
6. Held-out evaluation

Start at **50B training tokens**. Scale to 100B/300B only when validation supports it.

In [ ]:
!git clone -q https://github.com/D-engahmed/ATHLLM.git
%cd ATHLLM
!git checkout feat/spark-x25-frontier-rebuild
!pip install -q -r requirements.txt


In [ ]:
import torch, os, yaml
from pathlib import Path

print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM GB:', round(torch.cuda.get_device_properties(0).total_memory/1024**3, 2))

cfg = yaml.safe_load(Path('configs/distillation_4b_kaggle.yaml').read_text())
cfg


In [ ]:
# Architecture smoke test (uses a tiny configuration, not the full 4B model).
!python -m athllm.models.spark25


## Load the teacher locally

The teacher is used offline to generate training examples. Do **not** attempt to load a 397B teacher into a typical Kaggle runtime. Replace the model ID only with a checkpoint you are authorized to access.

In [ ]:
# Example only — run when the authorized teacher is accessible.
# !python -m athllm.tools.load_spark --model XHToken/Spark-X2.5-4B

TEACHER_MODEL = 'REPLACE_WITH_AUTHORIZED_TEACHER'
print('Teacher:', TEACHER_MODEL)


## Build a verified distillation manifest

Put approved source documents under `data/raw`. The manifest builder performs basic quality filtering and exact deduplication. Production training should add near-deduplication, provenance/license checks, contamination detection, language/domain balancing, and independent verification.

In [ ]:
from pathlib import Path
Path('data/raw').mkdir(parents=True, exist_ok=True)
Path('data/manifests').mkdir(parents=True, exist_ok=True)
print('Place authorized raw data in data/raw before running the manifest builder.')
# !python -m athllm.data.build_manifest --input data/raw --output data/manifests/train.jsonl --min-quality 0.85


## Training policy

Use the notebook to launch or monitor the student experiments. The intended sequence is:

`teacher distillation → reasoning SFT → coding/agent SFT → verifiable RL → self-improvement RL → int4 quantization → held-out evaluation`

In [ ]:
# Print the reproducible experiment configuration.
print(yaml.safe_dump(cfg, sort_keys=False))


## Quantization checkpoint

Quantization is the deployment step, not a substitute for distillation. Compare the quantized checkpoint against the BF16/FP16 student on the same held-out suites and record capability loss before selecting int4 for Kaggle inference.

In [ ]:
# Placeholder for the selected quantization backend.
# Keep calibration data held out from training and include reasoning + coding examples.
print('Next: run calibrated int4 quantization and compare against the unquantized student.')


## Reproducibility record

For every run, record: Git commit, dataset manifest hash/version, tokenizer version, model checkpoint, GPU type, context length, batch/gradient settings, seed, quantization settings, benchmark version, and evaluator version.

In [ ]:
import subprocess, json
commit = subprocess.check_output(['git','rev-parse','HEAD']).decode().strip()
print(json.dumps({'git_commit': commit, 'cuda': torch.version.cuda, 'torch': torch.__version__}, indent=2))
